In [ ]:
import os

# path setting
current = os.getcwd()
while os.path.basename(current) != "Data_center_and_fossil_energy_Replication":
    current = os.path.dirname(current)

BASE_PATH = current

RAW = os.path.join(BASE_PATH, 'Data', 'raw')
TEMP = os.path.join(BASE_PATH, 'Data', 'temp')
USE = os.path.join(BASE_PATH, 'Data', 'use')
FIGURES = os.path.join(BASE_PATH, 'Results', 'Figures')
TABLES = os.path.join(BASE_PATH, 'Results', 'Tables')

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    os.makedirs(path, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

def clean_column_names(df):
    """Clean column names to ensure Stata compatibility"""
    
    def clean_name(name):
        name = str(name)
        name = re.sub(r'[^\w\s]', '_', name)
        name = re.sub(r'\s+', '_', name)
        name = re.sub(r'_+', '_', name)
        name = name.strip('_')
        
        if name and name[0].isdigit():
            name = 'var_' + name
        
        if len(name) > 32:
            name = name[:32]
        
        if not name:
            name = 'unnamed_var'
            
        return name
    
    df.columns = [clean_name(col) for col in df.columns]
    
    # Handle duplicate column names
    seen = {}
    new_columns = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_col = f"{col}_{seen[col]}"
            if len(new_col) > 32:
                new_col = f"{col[:28]}_{seen[col]}"
            new_columns.append(new_col)
        else:
            seen[col] = 0
            new_columns.append(col)
    
    df.columns = new_columns
    return df

def clean_string_data(df):
    """Clean string data and handle encoding issues"""
    
    def clean_string(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            try:
                x.encode('latin-1')
                return x
            except UnicodeEncodeError:
                x = unicodedata.normalize('NFKD', x)
                x = ''.join(c for c in x if not unicodedata.combining(c))
                x = ''.join(char for char in x if ord(char) < 256)
                return x
        return x
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(clean_string)
    
    return df

# Read data
coal_units = pd.read_excel(os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"), sheet_name="Units")
coal_country = pd.read_excel(os.path.join(RAW, "Global-Coal-Plant-Tracker-January-2025.xlsx"), sheet_name="country")

# Select variables to merge
country_vars = coal_country[['Country/Area', 'Alpha-2 code', 'Alpha-3 code', 'Numeric', 'Region']]

# Merge data
coal_final = coal_units.merge(country_vars, on='Country/Area', how='left')

# Clean column names (using the method from the reference code)
coal_final = clean_column_names(coal_final)

# Clean string data
coal_final = clean_string_data(coal_final)

# Handle infinite values in numeric columns
numeric_columns = coal_final.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    coal_final[col] = coal_final[col].replace([np.inf, -np.inf], np.nan)

# Save
try:
    coal_final.to_stata(os.path.join(TEMP, 'coal_plants_final.dta'), write_index=False, version=118)
    print(f"Data saved, total {len(coal_final)} rows")
except UnicodeEncodeError as e:
    print(f"Encoding issue detected: {e}")
    for col in coal_final.columns:
        if coal_final[col].dtype == 'object':
            coal_final[col] = coal_final[col].astype(str).apply(
                lambda x: x.encode('ascii', 'ignore').decode('ascii') if x != 'nan' else np.nan
            )
    coal_final.to_stata(os.path.join(TEMP, 'coal_plants_final.dta'), write_index=False, version=118)
    print(f"Data saved (after ASCII cleaning)")

coal_final.to_csv(os.path.join(TEMP, 'coal_plants_final.csv'), index=False, encoding='utf-8')
print("CSV backup saved")
